1. Environment and Infrastructure Setup

In [ ]:
from google.colab import userdata

# 1. Configuration
USER_NAME = "mervynzwkoh"
REPO_NAME = "gout-transcriptome-causal"
TOKEN = userdata.get("GithubToken")

# 2. Setup the Git URL
# This tells Colab how to talk to your repo securely
repo_url = f"https://{TOKEN}@github.com/{USER_NAME}/{REPO_NAME}.git"

# 3. Clone the repository
!git clone {repo_url}

# 4. Move into the repo folder
%cd {REPO_NAME}

# 5. Configure Git identity (for your commit history)
!git config --global user.email "mervynzwkoh@gmail.com"
!git config --global user.name "Mervyn Koh"

In [ ]:
import os

# Create the folders we planned
folders = ['data', 'notebooks', 'src', 'docs']
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    # Create a dummy file in each so Git notices the folder
    with open(f"{folder}/.gitkeep", "w") as f:
        pass

# Integrate remote changes first, then add, commit and push
!git pull origin main --rebase
!git add .
!git commit -m "Initial commit: Established project folder structure and added data filtering logic"
!git push origin main

2. Data Ingestion and Filtering

In [ ]:
from datasets import load_dataset

# We use streaming=True so we don't run out of memory
dataset_stream = load_dataset("Xaira-Therapeutics/X-Atlas-Orion", streaming=True, split="HEK293T")

# Let's peek at the first "row" (cell) to see what information it contains
first_cell = next(iter(dataset_stream))
print(first_cell.keys())

In [ ]:
target_genes = ["ABCG2", "SLC22A12"]
filtered_cells = []

# We'll grab the first 100 cells that match our criteria for this test
print("Searching for gout-related perturbations...")

for cell in dataset_stream:
    # 'perturbation_name' or similar key in Orion identifies the targeted gene
    if cell['gene_target'] in target_genes:
        filtered_cells.append(cell)
        print(f"Found {len(filtered_cells)} cells so far...")

    if len(filtered_cells) >= 35:
        break

print(f"Success! Found {len(filtered_cells)} cells for initial analysis.")

3. Pre-processing Raw Data for Geneformer

In [ ]:
# Software Engineering Tip: We are 'Pre-processing' the data
# to make it ready for the Machine Learning model.

import pandas as pd

def format_for_geneformer(cell_data):
    """
    Combines token IDs and expression values into a ranked list.
    """
    tokens = cell_data['gene_token_id']
    counts = cell_data['gene_expression']

    # We create a dataframe for this specific cell's expression
    cell_df = pd.DataFrame({'token': tokens, 'count': counts})

    # Geneformer often uses rank-based normalization
    cell_df = cell_df.sort_values(by='count', ascending=False)

    return cell_df['token'].tolist()

# Example: Format the first cell we found
formatted_input = format_for_geneformer(filtered_cells[0])
print(f"First cell formatted. Number of expressed genes: {len(formatted_input)}")

In [ ]:
# We will use 'num_features' as our initial 'dose' proxy
# to see if more guides correlate with a stronger transcriptomic shift.

doses = [cell['num_features'] for cell in filtered_cells]
print(f"Dose distribution (Number of guides): {set(doses)}")

In [ ]:
# Move into your repository folder
%cd /content/gout-transcriptome-causal

# STEP 1: Vacuum the notebook metadata (Removes the 'widgets' and 'state' errors)
!nbstripout notebooks/01_data_ingestion.ipynb

# STEP 2: The standard Git sequence
!git add notebooks/01_data_ingestion.ipynb
!git commit -m "Fix: Stripped notebook metadata to resolve GitHub rendering error"
!git push origin feature/data-ingestion